# Mammographic Mass Classification

961 mammographic masses, each described by BI-RADS assessment, patient age, mass shape, margin and density. Predict severity — benign (0) or malignant (1). This is a real clinical dataset with missing values, and it is educational only, not a medical tool.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Mammographic Mass (id 161)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/161/data.csv')))
print(df.shape)
print('Masses per severity (0 = benign, 1 = malignant):')
print(df['Severity'].value_counts())
print('Missing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
df.head(3)


(961, 6)
Masses per severity (0 = benign, 1 = malignant):
Severity
0    516
1    445
Name: count, dtype: int64
Missing values per column:
BI-RADS     2
Age         5
Shape      31
Margin     48
Density    76
dtype: int64


,BI-RADS,Age,Shape,Margin,Density,Severity
0,5.0,67.0,3.0,5.0,3.0,1
1,4.0,43.0,1.0,1.0,NaN,1
2,5.0,58.0,4.0,5.0,3.0,1


In [2]:

feats = [c for c in df.columns if c != 'Severity']
X = df[feats]; y = df['Severity']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
# Missing values (162 across the feature columns): fill with the TRAINING
# median per column, so no test information leaks.
med = Xtr.median()
Xtr = Xtr.fillna(med); Xte = Xte.fillna(med)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.7842
Logistic Regression accuracy: 0.8174
              precision    recall  f1-score   support

           0       0.85      0.73      0.78       129
           1       0.73      0.85      0.79       112

    accuracy                           0.78       241
   macro avg       0.79      0.79      0.78       241
weighted avg       0.79      0.78      0.78       241



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Which mammographic features matter most?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Age        0.323187
BI-RADS    0.318673
Margin     0.176705
Shape      0.163361
Density    0.018073
dtype: float64


Winner: Logistic Regression
